# Prognoza volatilității → backtest vol-targeting pe SPY (vectorbt)

**Întrebarea:** prognozele mai bune ale volatilității duc la o strategie de tranzacționare mai bună? Și aduce un LSTM ceva în plus față de HAR și GARCH, atunci când experimentul este construit corect?

**Pașii**
1. Date zilnice OHLCV pentru SPY din 2000 (ajustate pentru dividende) și primele vizualizări.
2. Cinci estimatori ai volatilității realizate: close-to-close, Parkinson, Garman–Klass, Rogers–Satchell, Yang–Zhang.
3. O **grilă modele × estimatori** de prognoze, pe orizonturi de 1/5/20 zile:
   persistență naivă · HAR-RV · GARCH(1,1)-t · GJR-GARCH(1,1,1)-t · LSTM (direct) · LSTM (rezidual față de HAR).
4. Evaluarea prognozelor pe perioada de test, ținută deoparte până la final, cu teste care demască un model care doar „copiază ziua de ieri”.
5. Backtest în **vectorbt**: vol-targeting pe SPY, cu poziția dimensionată de fiecare prognoză, plus o strategie long/short pe medii mobile 50/200 ca reper.

**Reguli împotriva privirii în viitor (fiecare e verificată de `tests/test_no_lookahead.py`)**

| Unde | Protecție |
|---|---|
| Împărțirea datelor | antrenare ≤ 2018 · validare 2019–2022 · test 2023→azi. Toate cifrele raportate sunt doar din test. |
| Ținte (labels) | ținta(t,h) folosește doar zilele t+1…t+h; rândurile a căror fereastră-țintă trece peste o graniță între perioade sunt **eliminate (embargo)**. |
| HAR | reestimat la fiecare 21 de zile pe o fereastră extinsă, folosind doar ținte deja observate la data reestimării. |
| GARCH | reestimat la fiecare 21 de zile pe randamentele de până atunci; parametrii sunt apoi înghețați, iar recursia varianței avansează câte o zi. |
| LSTM | scalarea, normalizarea țintei și ponderile rețelei sunt estimate doar pe antrenare; validarea servește doar la oprirea timpurie (early stopping); antrenat o singură dată. |
| Backtest | ponderile decise la închiderea zilei *t* sunt executate la **deschiderea zilei t+1**, cu comisioane și slippage. |

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)

from backtester_demo import config, data, pipeline, evaluation, backtest, plots
from backtester_demo.volatility import all_daily_variances
from backtester_demo.splits import split_labels

df = data.load_ohlcv()          # cached in data/spy.parquet; pass refresh=True to re-download
print(f"{config.TICKER}: {len(df)} zile, {df.index[0].date()} → {df.index[-1].date()}")
print(split_labels(df.index).value_counts().reindex(["train", "val", "test"]).to_string())
df.tail()

SPY: 6749 zile, 2000-01-03 → 2026-11-02
split
train    4779
val      1008
test      962


,Open,High,Low,Close,Volume
Date,,,,,
2026-10-27,767.340027,771.140015,765.210022,769.659973,28631700.0
2026-10-28,771.429993,773.359985,769.239990,772.309998,30133800.0
2026-10-29,768.789978,770.760010,766.619995,769.359985,40299600.0
2026-10-30,761.380005,761.460022,750.359985,755.719971,77696500.0
2026-11-02,759.450012,763.859985,751.599976,752.099976,50668800.0


## 1 · Datele OHLC
Tot istoricul, cu perioadele de antrenare / validare / test colorate diferit, apoi un zoom pe ultimul an.

In [2]:
plots.ohlc_chart(df).show()

In [3]:
plots.ohlc_chart(df, start=(df.index[-1] - pd.DateOffset(years=1)).strftime('%Y-%m-%d')).show()

Randamentele au „cozi groase” (valori extreme mai dese decât la distribuția normală), iar volatilitatea vine în grupuri (clustering): zilele agitate urmează după zile agitate. Acest clustering este singurul motiv pentru care volatilitatea se poate prognoza.

In [4]:
plots.returns_overview(df).show()

## 2 · „Spectrul” volatilității: cinci estimatori

| Estimator | Varianța zilnică | Ce folosește |
|---|---|---|
| Close-to-close | $r_t^2$ | doar închiderea, **include** saltul peste noapte |
| Parkinson | $\ln(H/L)^2 / 4\ln 2$ | intervalul max–min, doar intraday |
| Garman–Klass | $\tfrac12\ln(H/L)^2-(2\ln2-1)\ln(C/O)^2$ | OHLC, doar intraday |
| Rogers–Satchell | $\ln\tfrac HO\ln\tfrac HC+\ln\tfrac LO\ln\tfrac LC$ | OHLC, robust la trend (drift) |
| Yang–Zhang | overnight² + k·deschidere-închidere² + (1−k)·RS | OHLC, **include** saltul peste noapte |

Contează două lucruri mai departe. Primul este **nivelul**: estimatorii doar intraday indică o volatilitate cu ~17% mai mică decât close-to-close, pentru că ratează saltul de peste noapte. La vol-targeting trebuie convertiți înapoi în unități close-to-close, altfel strategia își asumă prea mult risc. Al doilea este **zgomotul**: un singur randament zilnic la pătrat este o măsurătoare foarte zgomotoasă, pe când estimatorii bazați pe interval (range) sunt mult mai stabili. Se vede din autocorelația de lag 1 a log-volatilității.

In [5]:
V = all_daily_variances(df)
display(plots.estimator_table(V))
plots.estimators_chart(V).show()
plots.estimator_corr(V).show()

,mean_ann_vol_%,level_vs_cc,autocorr_lag1_logvol,std_logvol
close_to_close,19.236,1.000,0.141,1.237
parkinson,15.778,0.820,0.664,0.622
garman_klass,15.909,0.827,0.685,0.605
rogers_satchell,16.109,0.837,0.597,0.641
yang_zhang,19.553,1.016,0.648,0.613


## 3 · Grila de prognoze

Grila completă (6 modele × 5 estimatori × 3 orizonturi) durează ~6 minute pe CPU și este salvată în `artifacts/`.

**De ce două LSTM-uri?** O rețea care prezice direct log-volatilitatea de mâine, primind ca intrare volatilitatea de azi, își poate minimiza eroarea (MSE) dând pur și simplu o copie netezită a zilei de azi. Graficul arată foarte bine, dar nu conține nicio informație în plus față de modelul naiv. `lstm_residual` prezice în schimb `țintă − ancoră_HAR`, unde ancora HAR este estimată doar pe antrenare. Persistența este deja în ancoră, așa că, dacă rețeaua nu învață nimic, ea revine la HAR, nu la o copie lag-1. `lstm_direct` este păstrat ca grup de control. Diagnosticele de mai jos arată dacă vreunul a căzut în capcană.

GARCH modelează randamentele close-to-close. Prognoza lui este transpusă pe fiecare estimator printr-o calibrare liniară estimată **doar pe perioada de validare**.

In [6]:
grid = pipeline.forecast_grid(df)
tgts = pipeline.targets(V)
metrics = pipeline.metrics_table(grid, tgts)
print(len(grid), "serii de prognoze;", len(metrics), "rânduri de metrici")

30 serii de prognoze; 90 rânduri de metrici


## 4 · Evaluarea prognozelor (doar perioada de test)

- **skill_vs_naive** = 1 − MSE/MSE(naiv). Zero sau mai puțin înseamnă că modelul nu aduce nimic peste „mâine = azi”.
- **change_corr** = corr(prognoză − naiv, realizat − naiv). Prezice modelul *mișcarea* volatilității? O copie lag-1 obține ≈ 0.
- **dm_vs_har**: statistica Diebold–Mariano pe pierderea QLIKE, comparativ cu HAR. Negativ = mai bun decât HAR; |DM| > 1,96 este semnificativ la 5%.
- **QLIKE** penalizează subestimarea volatilității mai mult decât supraestimarea, ceea ce reflectă costul asimetric din managementul riscului.

In [7]:
for h in config.HORIZONS:
    plots.metric_heatmap(metrics, "skill_vs_naive", h).show()

In [8]:
plots.metric_heatmap(metrics, "change_corr", 1).show()
plots.metric_heatmap(metrics, "dm_vs_har", config.TRADE_HORIZON, fmt=".1f", reverse=True).show()
plots.metric_heatmap(metrics, "qlike", config.TRADE_HORIZON, reverse=True).show()

In [9]:
gk = metrics[metrics.estimator == "garman_klass"].set_index(["horizon", "model"]).drop(columns="estimator")
gk.round(3).style.background_gradient(subset=["skill_vs_naive", "change_corr"], cmap="RdYlGn").format(precision=3)

,,rmse,qlike,skill_vs_naive,change_corr,dm_vs_har,dm_p,mz_a,mz_b,mz_r2
horizon,model,,,,,,,,,
1,naive,0.469,0.606,0.000,nan,6.095,0.000,-2.473,0.525,0.277
5,naive,0.437,0.581,0.000,nan,8.178,0.000,-2.986,0.409,0.279
20,naive,0.486,0.822,0.000,nan,6.591,0.000,-3.766,0.248,0.150
1,har,0.395,0.394,0.293,0.543,0.000,1.000,-0.569,0.894,0.334
5,har,0.311,0.234,0.494,0.689,0.000,1.000,-1.201,0.770,0.346
20,har,0.293,0.198,0.636,0.785,0.000,1.000,-2.020,0.606,0.228
1,garch,0.421,0.424,0.196,0.538,1.554,0.120,-0.803,0.847,0.245
5,garch,0.322,0.248,0.455,0.701,0.789,0.430,-1.021,0.807,0.283
20,garch,0.305,0.198,0.606,0.791,-0.046,0.963,-1.521,0.714,0.177


### Copiază vreun model ziua de ieri?

**Profilul de lag:** corr(prognoza făcută la *t*, log-volatilitatea zilnică realizată la *t+k*). Toate prognozele sunt construite din informații de până la *t*, deci toate corelează puternic cu trecutul (k ≤ 0). Modelul naiv coincide cu valoarea de la k=0. Ce contează este partea **k > 0**: un model care chiar „vede înainte” trebuie să-l bată pe cel naiv acolo.

In [10]:
lv1 = 0.5 * np.log(V["garman_klass"])
mask = pipeline.test_mask(df.index)
prof = pd.DataFrame({m: evaluation.lag_profile(grid[(m, "garman_klass")].loc[mask, 1], lv1, 5)
                     for m in pipeline.MODELS})
plots.lag_profile_chart(prof).show()
prof.loc[[1, 2, 5]].round(3)

,naive,har,garch,gjr_garch,lstm_direct,lstm_residual
1,0.526,0.578,0.495,0.589,0.639,0.640
2,0.460,0.514,0.453,0.539,0.568,0.570
5,0.334,0.378,0.339,0.390,0.403,0.401


In [11]:
plots.forecast_vs_realised(grid, tgts, "garman_klass", config.TRADE_HORIZON,
                           ["naive", "har", "garch", "lstm_residual"]).show()

## 5 · Backtest cu vectorbt

**Vol targeting (țintirea volatilității).** La fiecare închidere, ponderea în SPY este
$w_t = \min\!\big(1,\ \sigma^* \cdot \kappa_e / \hat\sigma_{t,h=5}\big)$, cu $\sigma^* = 12\%$ anualizat. Aici $\kappa_e$ este nivelul estimatorului raportat la close-to-close, măsurat pe antrenare. Ponderea se execută la deschiderea următoare, cu 2 bps comision și 3 bps slippage.
Motorul open-source vectorbt nu poate împrumuta cash pentru poziții long, așa că efectul de levier este limitat la 1×.

**Repere:** buy & hold; un long/short pe medii mobile 50/200 (+1/−1, parametri fixați dinainte, neoptimizați); același long/short scalat cu prognoza de volatilitate a fiecărui model; și un **ORACLE** care dimensionează poziția cu volatilitatea *viitoare realizată*. Oracle-ul privește intenționat în viitor și arată plafonul pentru un prognozator perfect.

Ce ar trebui să aducă o prognoză bună a volatilității: **volatilitate realizată stabilă, aproape de țintă** (vol-of-vol mic), drawdown-uri mai mici și un Sharpe mai bun decât o estimare învechită.

In [12]:
W = pipeline.strategy_weights(df, grid, V, tgts)
pf = backtest.run(df, W)
summary = backtest.summary(pf)
summary.sort_values("sharpe", ascending=False).round(2).head(15)

,total_return_%,cagr_%,ann_vol_%,sharpe,sortino,max_dd_%,vol_of_vol_%,trades,calmar
vt|ORACLE|yang_zhang,132.68,24.76,10.70,2.12,3.56,-9.50,0.98,478,2.61
vt|ORACLE|parkinson,139.82,25.75,11.10,2.12,3.56,-9.54,1.08,410,2.70
vt|ORACLE|garman_klass,140.97,25.91,11.23,2.11,3.53,-9.54,1.19,394,2.72
vt|ORACLE|close_to_close,121.82,23.21,10.32,2.08,3.52,-7.67,0.81,505,3.03
vt|ORACLE|rogers_satchell,138.78,25.61,11.40,2.06,3.42,-9.70,1.36,385,2.64
vt|har|garman_klass,80.62,16.75,11.89,1.36,1.92,-13.22,1.66,432,1.27
vt|har|rogers_satchell,81.54,16.91,12.07,1.36,1.90,-13.57,1.86,424,1.25
vt|naive|garman_klass,78.17,16.33,11.76,1.35,1.89,-12.95,1.61,480,1.26
buy_hold,105.04,20.69,14.86,1.34,1.85,-18.76,5.48,1,1.10
vt|har|parkinson,78.20,16.34,11.86,1.34,1.88,-13.25,1.60,437,1.23


In [13]:
plots.summary_heatmap(summary, "sharpe").show()
plots.summary_heatmap(summary, "vol_of_vol_%", reverse=True).show()
plots.summary_heatmap(summary, "max_dd_%").show()

In [14]:
show = ["buy_hold", "vt|naive|garman_klass", "vt|har|garman_klass", "vt|garch|garman_klass",
        "vt|lstm_residual|garman_klass", "vt|ORACLE|garman_klass"]
plots.equity_curves(pf, show).show()
plots.rolling_vol_vs_target(pf, show).show()
plots.drawdowns(pf, show).show()
plots.weights_chart(W, show[1:5]).show()

### Long/short simplu vs long/short scalat cu volatilitatea

In [15]:
ls = ["buy_hold", "sma_long_short"] + [c for c in summary.index if c.startswith("sma_ls_vt|")]
display(summary.loc[ls].round(2))
plots.equity_curves(pf, ls, title="SMA 50/200 long/short: raw vs vol-scaled").show()

,total_return_%,cagr_%,ann_vol_%,sharpe,sortino,max_dd_%,vol_of_vol_%,trades,calmar
buy_hold,105.04,20.69,14.86,1.34,1.85,-18.76,5.48,1,1.10
sma_long_short,33.50,7.86,14.90,0.58,0.79,-26.96,5.41,68,0.29
sma_ls_vt|naive|garman_klass,27.63,6.60,11.81,0.60,0.85,-22.31,1.54,495,0.30
sma_ls_vt|har|garman_klass,36.14,8.42,11.92,0.74,1.04,-20.59,1.60,443,0.41
sma_ls_vt|garch|garman_klass,33.17,7.79,12.44,0.67,0.92,-21.69,2.07,426,0.36
sma_ls_vt|gjr_garch|garman_klass,30.79,7.28,12.02,0.65,0.90,-21.67,1.69,464,0.34
sma_ls_vt|lstm_direct|garman_klass,32.79,7.71,11.96,0.68,0.95,-22.09,1.61,432,0.35
sma_ls_vt|lstm_residual|garman_klass,34.36,8.04,12.00,0.70,0.99,-22.34,1.69,411,0.36


### Statisticile vectorbt pentru o strategie

In [16]:
pf["vt|lstm_residual|garman_klass"].stats()

Start                              2023-01-03 00:00:00
End                                2026-11-02 00:00:00
Period                               962 days 00:00:00
Start Value                                   100000.0
End Value                                178128.301494
Total Return [%]                             78.128301
Benchmark Return [%]                        107.057727
Max Gross Exposure [%]                           100.0
Total Fees Paid                             717.198471
Max Drawdown [%]                              14.27844
Max Drawdown Duration                119 days 00:00:00
Total Trades                                       183
Total Closed Trades                                182
Total Open Trades                                    1
Open Trade PnL                             5010.816891
Win Rate [%]                                 81.318681
Best Trade [%]                               23.722219
Worst Trade [%]                             -14.044351
Avg Winnin

## 6 · Concluzii (calculate din această rulare)

In [17]:
vt = summary[summary.index.str.startswith("vt|") & ~summary.index.str.contains("ORACLE")]
best = vt["sharpe"].idxmax()
by_model = vt.groupby(vt.index.str.split("|").str[1])[["sharpe", "vol_of_vol_%", "max_dd_%"]].mean()
by_est = vt.groupby(vt.index.str.split("|").str[2])[["sharpe", "vol_of_vol_%", "max_dd_%"]].mean()
h = config.TRADE_HORIZON
sk = metrics[metrics.horizon == h].groupby("model")["skill_vs_naive"].mean().sort_values(ascending=False)
print(f"Cea mai bună strategie vol-target după Sharpe: {best}  ({vt.loc[best, 'sharpe']:.2f} vs buy&hold {summary.loc['buy_hold', 'sharpe']:.2f})")
print(f"\nSkill mediu față de naiv la h={h}, peste toți estimatorii:\n{sk.round(3).to_string()}")
print("\nBacktest: medie peste estimatori, pe model:"); display(by_model.round(2).sort_values("sharpe", ascending=False))
print("Backtest: medie peste modele, pe estimator:"); display(by_est.round(2).sort_values("sharpe", ascending=False))

Cea mai bună strategie vol-target după Sharpe: vt|har|garman_klass  (1.36 vs buy&hold 1.34)

Skill mediu față de naiv la h=5, peste toți estimatorii:
model
lstm_direct      0.669
lstm_residual    0.665
gjr_garch        0.650
har              0.602
garch            0.586
naive            0.000

Backtest: medie peste estimatori, pe model:


,sharpe,vol_of_vol_%,max_dd_%
har,1.32,1.73,-13.51
gjr_garch,1.31,1.70,-13.81
lstm_residual,1.31,1.75,-14.29
lstm_direct,1.31,1.71,-14.26
garch,1.28,2.06,-14.89
naive,1.23,1.91,-14.75


Backtest: medie peste modele, pe estimator:


,sharpe,vol_of_vol_%,max_dd_%
garman_klass,1.32,1.77,-13.97
rogers_satchell,1.32,1.90,-14.23
parkinson,1.31,1.74,-14.05
yang_zhang,1.27,1.57,-13.71
close_to_close,1.25,2.07,-15.30


**Precauții**
- Perioada de test (2023 → azi) este o singură traiectorie a pieței. Diferențele de câteva sutimi de Sharpe sunt în limita zgomotului; metricile de prognoză, cu mii de observații zilnice, sunt mult mai de încredere decât o singură curbă de capital.
- LSTM-ul este antrenat o singură dată, pe date până în 2018. Într-un sistem real ar fi reantrenat walk-forward, așa cum se face deja pentru HAR și GARCH.
- Dimensionarea după estimatori doar intraday ignoră riscul de peste noapte dacă nu este renivelată cu κ. Yang–Zhang și close-to-close includ nativ saltul de peste noapte.